In [97]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [98]:
import pandas as pd
import numpy as np

# [1] air 데이터셋 전처리(weather의 시간단위와 맞추기 위함)

In [99]:
# 'air_2021.csv'파일을 'air_21' 변수에, 'air_2022.csv'파일을 'air_22' 변수에 저장
air_21 = pd.read_csv('/content/drive/MyDrive/AI+X/AI+X선도인재양성프로젝트(중급)/(제공용) 미세먼지 교안, 데이터 및 실습 과제/11주차/air_2021.csv',index_col=0)
air_22 = pd.read_csv('/content/drive/MyDrive/AI+X/AI+X선도인재양성프로젝트(중급)/(제공용) 미세먼지 교안, 데이터 및 실습 과제/11주차/air_2022.csv',index_col=0)

In [100]:
air_21.head(3)

,지역,망,측정소코드,측정소명,측정일시,SO2,CO,O3,NO2,PM10,PM25,주소
0,서울 종로구,도시대기,111123,종로구,2021100101,0.003,0.6,0.002,0.039,31.0,18.0,서울 종로구 종로35가길 19
1,서울 종로구,도시대기,111123,종로구,2021100102,0.003,0.6,0.002,0.035,27.0,16.0,서울 종로구 종로35가길 19
2,서울 종로구,도시대기,111123,종로구,2021100103,0.003,0.6,0.002,0.033,28.0,18.0,서울 종로구 종로35가길 19


In [101]:
air_21['time'] = air_21['측정일시'].astype(str)

# 자정은 다음날 00시여야 하는데 당일날 24시로 되어 있음...
temp = air_21['time'].str[-2:] == '24'


air_21.loc[temp, 'time'] = ( pd.to_datetime(air_21.loc[temp, 'time'].str.slice(0,8), format='%Y%m%d') + pd.Timedelta(days=1)).dt.strftime('%Y%m%d') + '00'

air_21['time'] = pd.to_datetime(air_21['time'], format='%Y%m%d%H')


In [102]:
air_22['time'] = air_22['측정일시'].astype(str)

# 자정은 다음날 00시여야 하는데 당일날 24시로 되어 있음...
temp = air_22['time'].str[-2:] == '24'


air_22.loc[temp, 'time'] = (pd.to_datetime(air_22.loc[temp, 'time'].str.slice(0,8), format='%Y%m%d') + pd.Timedelta(days=1)).dt.strftime('%Y%m%d') + '00'

air_22['time'] = pd.to_datetime(air_22['time'], format='%Y%m%d%H')


In [103]:
 air_21.head(3)

,지역,망,측정소코드,측정소명,측정일시,SO2,CO,O3,NO2,PM10,PM25,주소,time
0,서울 종로구,도시대기,111123,종로구,2021100101,0.003,0.6,0.002,0.039,31.0,18.0,서울 종로구 종로35가길 19,2021-10-01 01:00:00
1,서울 종로구,도시대기,111123,종로구,2021100102,0.003,0.6,0.002,0.035,27.0,16.0,서울 종로구 종로35가길 19,2021-10-01 02:00:00
2,서울 종로구,도시대기,111123,종로구,2021100103,0.003,0.6,0.002,0.033,28.0,18.0,서울 종로구 종로35가길 19,2021-10-01 03:00:00


In [104]:
# 2022년 1월 1일 00시 데이터가 2021데이터에 있음.
air_21[air_21['time']=='2022-01-01 00:00:00']

,지역,망,측정소코드,측정소명,측정일시,SO2,CO,O3,NO2,PM10,PM25,주소,time
2207,서울 종로구,도시대기,111123,종로구,2021123124,0.003,0.4,0.026,0.016,20.0,9.0,서울 종로구 종로35가길 19,2022-01-01


In [105]:
# 2022년 1월 1일 00시 데이터가 2021데이터에 있음. -> 수정
air_22 = pd.concat([air_22, air_21[air_21['time']=='2022-01-01 00:00:00']], ignore_index=True)

In [106]:
air_21 = air_21[air_21['time']!='2022-01-01 00:00:00']

In [107]:
air_22[air_22['time']=='2022-01-01 00:00:00']

,지역,망,측정소코드,측정소명,측정일시,SO2,CO,O3,NO2,PM10,PM25,주소,time
2160,서울 종로구,도시대기,111123,종로구,2021123124,0.003,0.4,0.026,0.016,20.0,9.0,서울 종로구 종로35가길 19,2022-01-01


In [108]:
air_21 = air_21.sort_values(by='time')
air_22 = air_22.sort_values(by='time')

In [109]:
air_21.head(3)

,지역,망,측정소코드,측정소명,측정일시,SO2,CO,O3,NO2,PM10,PM25,주소,time
2208,서울 종로구,도시대기,111123,종로구,2021010101,0.002,0.5,0.022,0.016,24.0,14.0,서울 종로구 종로35가길 19,2021-01-01 01:00:00
2209,서울 종로구,도시대기,111123,종로구,2021010102,0.002,0.6,0.018,0.020,25.0,14.0,서울 종로구 종로35가길 19,2021-01-01 02:00:00
2210,서울 종로구,도시대기,111123,종로구,2021010103,0.002,0.6,0.013,0.025,27.0,16.0,서울 종로구 종로35가길 19,2021-01-01 03:00:00


# [2] 날씨 데이터 불러오기

In [110]:
#'weather_2021.csv'파일을 'weather_21' 변수에, 'weather_2022.csv'파일을 'weather_22' 변수에 저장
weather_21 = pd.read_csv('/content/drive/MyDrive/AI+X/AI+X선도인재양성프로젝트(중급)/(제공용) 미세먼지 교안, 데이터 및 실습 과제/11주차/weather_2021.csv', sep=',', encoding='cp949')
weather_22 = pd.read_csv('/content/drive/MyDrive/AI+X/AI+X선도인재양성프로젝트(중급)/(제공용) 미세먼지 교안, 데이터 및 실습 과제/11주차/weather_2022.csv', sep=',', encoding='cp949')

In [111]:
weather_21.head(3)

,지점,지점명,일시,기온(°C),기온 QC플래그,강수량(mm),강수량 QC플래그,풍속(m/s),풍속 QC플래그,풍향(16방위),...,최저운고(100m ),시정(10m),지면상태(지면상태코드),현상번호(국내식),지면온도(°C),지면온도 QC플래그,5cm 지중온도(°C),10cm 지중온도(°C),20cm 지중온도(°C),30cm 지중온도(°C)
0,108,서울,2021-01-01 01:00,-8.7,NaN,NaN,NaN,2.4,NaN,270.0,...,NaN,2000,NaN,NaN,-6.9,NaN,-1.0,-0.8,0.3,1.6
1,108,서울,2021-01-01 02:00,-9.1,NaN,NaN,NaN,1.6,NaN,270.0,...,NaN,2000,NaN,NaN,-7.1,NaN,-1.1,-0.8,0.3,1.6
2,108,서울,2021-01-01 03:00,-9.3,NaN,NaN,NaN,1.1,NaN,250.0,...,NaN,2000,NaN,NaN,-7.3,NaN,-1.2,-0.9,0.3,1.6


In [112]:
weather_21['time'] = pd.to_datetime(weather_21['일시'])
weather_22['time'] = pd.to_datetime(weather_22['일시'])

In [113]:
weather_21.head(3)

,지점,지점명,일시,기온(°C),기온 QC플래그,강수량(mm),강수량 QC플래그,풍속(m/s),풍속 QC플래그,풍향(16방위),...,시정(10m),지면상태(지면상태코드),현상번호(국내식),지면온도(°C),지면온도 QC플래그,5cm 지중온도(°C),10cm 지중온도(°C),20cm 지중온도(°C),30cm 지중온도(°C),time
0,108,서울,2021-01-01 01:00,-8.7,NaN,NaN,NaN,2.4,NaN,270.0,...,2000,NaN,NaN,-6.9,NaN,-1.0,-0.8,0.3,1.6,2021-01-01 01:00:00
1,108,서울,2021-01-01 02:00,-9.1,NaN,NaN,NaN,1.6,NaN,270.0,...,2000,NaN,NaN,-7.1,NaN,-1.1,-0.8,0.3,1.6,2021-01-01 02:00:00
2,108,서울,2021-01-01 03:00,-9.3,NaN,NaN,NaN,1.1,NaN,250.0,...,2000,NaN,NaN,-7.3,NaN,-1.2,-0.9,0.3,1.6,2021-01-01 03:00:00


# [3] air데이터셋과 weather데이터셋 병합

In [114]:
df_21 = pd.merge(air_21, weather_21, on='time', how='inner')
df_22 = pd.merge(air_22, weather_22, on='time', how='inner')

In [115]:
df_21.head(3)

,지역,망,측정소코드,측정소명,측정일시,SO2,CO,O3,NO2,PM10,...,최저운고(100m ),시정(10m),지면상태(지면상태코드),현상번호(국내식),지면온도(°C),지면온도 QC플래그,5cm 지중온도(°C),10cm 지중온도(°C),20cm 지중온도(°C),30cm 지중온도(°C)
0,서울 종로구,도시대기,111123,종로구,2021010101,0.002,0.5,0.022,0.016,24.0,...,NaN,2000,NaN,NaN,-6.9,NaN,-1.0,-0.8,0.3,1.6
1,서울 종로구,도시대기,111123,종로구,2021010102,0.002,0.6,0.018,0.020,25.0,...,NaN,2000,NaN,NaN,-7.1,NaN,-1.1,-0.8,0.3,1.6
2,서울 종로구,도시대기,111123,종로구,2021010103,0.002,0.6,0.013,0.025,27.0,...,NaN,2000,NaN,NaN,-7.3,NaN,-1.2,-0.9,0.3,1.6


In [116]:
df_21.columns

Index(['지역', '망', '측정소코드', '측정소명', '측정일시', 'SO2', 'CO', 'O3', 'NO2', 'PM10',
       'PM25', '주소', 'time', '지점', '지점명', '일시', '기온(°C)', '기온 QC플래그',
       '강수량(mm)', '강수량 QC플래그', '풍속(m/s)', '풍속 QC플래그', '풍향(16방위)', '풍향 QC플래그',
       '습도(%)', '습도 QC플래그', '증기압(hPa)', '이슬점온도(°C)', '현지기압(hPa)', '현지기압 QC플래그',
       '해면기압(hPa)', '해면기압 QC플래그', '일조(hr)', '일조 QC플래그', '일사(MJ/m2)',
       '일사 QC플래그', '적설(cm)', '3시간신적설(cm)', '전운량(10분위)', '중하층운량(10분위)',
       '운형(운형약어)', '최저운고(100m )', '시정(10m)', '지면상태(지면상태코드)', '현상번호(국내식)',
       '지면온도(°C)', '지면온도 QC플래그', '5cm 지중온도(°C)', '10cm 지중온도(°C)',
       '20cm 지중온도(°C)', '30cm 지중온도(°C)'],
      dtype='object')

# [4] 필요없는 변수 제거

In [117]:
# df_21, df_22에 사용할 변수들만 할당
df_21.drop(['지역', '망', '측정소코드', '측정소명', '측정일시', '주소', '지점', '지점명', '일시', '기온 QC플래그','강수량 QC플래그', '풍속 QC플래그', '풍향 QC플래그', '습도 QC플래그', '현지기압 QC플래그', '해면기압 QC플래그', '일조 QC플래그', '일사 QC플래그', '지면상태(지면상태코드)', '현상번호(국내식)', '지면온도 QC플래그', '운형(운형약어)'], axis=1, inplace=True)
df_22.drop(['지역', '망', '측정소코드', '측정소명', '측정일시', '주소', '지점', '지점명', '일시', '기온 QC플래그','강수량 QC플래그', '풍속 QC플래그', '풍향 QC플래그', '습도 QC플래그', '현지기압 QC플래그', '해면기압 QC플래그', '일조 QC플래그', '일사 QC플래그', '지면상태(지면상태코드)', '현상번호(국내식)', '지면온도 QC플래그', '운형(운형약어)'], axis=1, inplace=True)

In [118]:
df_21.columns

Index(['SO2', 'CO', 'O3', 'NO2', 'PM10', 'PM25', 'time', '기온(°C)', '강수량(mm)',
       '풍속(m/s)', '풍향(16방위)', '습도(%)', '증기압(hPa)', '이슬점온도(°C)', '현지기압(hPa)',
       '해면기압(hPa)', '일조(hr)', '일사(MJ/m2)', '적설(cm)', '3시간신적설(cm)', '전운량(10분위)',
       '중하층운량(10분위)', '최저운고(100m )', '시정(10m)', '지면온도(°C)', '5cm 지중온도(°C)',
       '10cm 지중온도(°C)', '20cm 지중온도(°C)', '30cm 지중온도(°C)'],
      dtype='object')

In [119]:
# time 변수를 index로 세팅
df_21 = df_21.set_index('time')
df_22 = df_22.set_index('time')

df_21 = df_21.sort_index()
df_22 = df_22.sort_index()

In [120]:
df_21

,SO2,CO,O3,NO2,PM10,PM25,기온(°C),강수량(mm),풍속(m/s),풍향(16방위),...,3시간신적설(cm),전운량(10분위),중하층운량(10분위),최저운고(100m ),시정(10m),지면온도(°C),5cm 지중온도(°C),10cm 지중온도(°C),20cm 지중온도(°C),30cm 지중온도(°C)
time,,,,,,,,,,,,,,,,,,,,,
2021-01-01 01:00:00,0.002,0.5,0.022,0.016,24.0,14.0,-8.7,NaN,2.4,270.0,...,NaN,0.0,0,NaN,2000,-6.9,-1.0,-0.8,0.3,1.6
2021-01-01 02:00:00,0.002,0.6,0.018,0.020,25.0,14.0,-9.1,NaN,1.6,270.0,...,NaN,0.0,0,NaN,2000,-7.1,-1.1,-0.8,0.3,1.6
2021-01-01 03:00:00,0.002,0.6,0.013,0.025,27.0,16.0,-9.3,NaN,1.1,250.0,...,NaN,0.0,0,NaN,2000,-7.3,-1.2,-0.9,0.3,1.6
2021-01-01 04:00:00,0.003,0.6,0.011,0.027,23.0,13.0,-9.3,NaN,0.3,0.0,...,NaN,0.0,0,NaN,2000,-7.5,-1.3,-1.0,0.2,1.5
2021-01-01 05:00:00,0.003,0.6,0.008,0.032,24.0,14.0,-9.7,NaN,1.9,20.0,...,NaN,0.0,0,NaN,2000,-7.6,-1.3,-1.0,0.2,1.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-31 19:00:00,0.003,0.4,0.026,0.019,25.0,10.0,-6.4,NaN,2.2,250.0,...,NaN,0.0,0,NaN,2000,-4.5,-0.5,-0.7,-0.1,1.0
2021-12-31 20:00:00,0.003,0.4,0.025,0.020,26.0,8.0,-6.3,NaN,4.1,320.0,...,NaN,0.0,0,NaN,2000,-5.3,-0.6,-0.8,-0.1,1.0
2021-12-31 21:00:00,0.003,0.4,0.030,0.014,27.0,9.0,-6.7,NaN,4.8,320.0,...,NaN,0.0,0,NaN,2000,-5.7,-0.7,-0.8,-0.1,1.0


# [5] 결측치 처리

In [121]:
# df_21의 결측치 확인
df_21.isna().sum()

,0
SO2,112
CO,80
O3,97
NO2,80
PM10,105
PM25,97
기온(°C),0
강수량(mm),7810
풍속(m/s),2
풍향(16방위),2


In [122]:
# df_21의 결측치 확인
df_22.isna().sum()

,0
SO2,21
CO,21
O3,21
NO2,21
PM10,38
PM25,25
기온(°C),0
강수량(mm),2079
풍속(m/s),0
풍향(16방위),0


In [123]:
# df_21, df_22의 변수 중 '강수량(mm)'의 결측치를 처리
# 0.0mm운 강수량 측정 최소단위인 0.1mm 미만을 의미. 공란은 무강수를 의미 -> NaN값을 0으로 채움.
df_21['강수량(mm)'] = df_21['강수량(mm)'].fillna(0)
df_22['강수량(mm)'] = df_22['강수량(mm)'].fillna(0)

In [124]:
df_21['최저운고(100m )'] = df_21['최저운고(100m )'].fillna(0)
df_22['최저운고(100m )'] = df_22['최저운고(100m )'].fillna(0)

In [125]:
df_21['적설(cm)'] = df_21['적설(cm)'].fillna(0)
df_21['3시간신적설(cm)'] = df_21['3시간신적설(cm)'].fillna(0)
df_21['일조(hr)'] = df_21['일조(hr)'].fillna(0)
df_21['일사(MJ/m2)'] = df_21['일사(MJ/m2)'].fillna(0)

df_22['적설(cm)'] = df_22['적설(cm)'].fillna(0)
df_22['3시간신적설(cm)'] = df_22['3시간신적설(cm)'].fillna(0)
df_22['일조(hr)'] = df_22['일조(hr)'].fillna(0)
df_22['일사(MJ/m2)'] = df_22['일사(MJ/m2)'].fillna(0)

In [126]:
df_21['전운량(10분위)'] = df_21['전운량(10분위)'].interpolate(method='linear')
df_21['풍속(m/s)'] = df_21['풍속(m/s)'].interpolate(method='linear')

df_21.loc['2021-01-07 11:00:00', '풍향(16방위)'] = 270.0
df_21.loc['2021-01-07 12:00:00', '풍향(16방위)'] = 290.0

In [127]:
ground_col = ['지면온도(°C)', '5cm 지중온도(°C)', '10cm 지중온도(°C)', '20cm 지중온도(°C)', '30cm 지중온도(°C)']

# NaN값 주변을 보면, 온도가 시간이 지날수록 내려가고 있음. -> 선형 보간
df_21[ground_col] = df_21[ground_col].interpolate(method='linear')

In [128]:
air_cols = ['SO2', 'CO', 'O3', 'NO2', 'PM10', 'PM25']

# NaN값 주변을 보면, 대기 농도가 시간이 지날수록 올라가거나 내려가고 있음. -> 선형 보간
df_21[air_cols] = df_21[air_cols].interpolate(method='linear')

# 미세먼지 농도는 소수부분이 없으므로 반올림
df_21['PM10'] = df_21['PM10'].round()
df_21['PM25'] = df_21['PM25'].round()

df_22[air_cols] = df_22[air_cols].interpolate(method='linear')

df_22['PM10'] = df_22['PM10'].round()
df_22['PM25'] = df_22['PM25'].round()

In [129]:
df_21.isna().sum()

,0
SO2,0
CO,0
O3,0
NO2,0
PM10,0
PM25,0
기온(°C),0
강수량(mm),0
풍속(m/s),0
풍향(16방위),0


In [130]:
df_22.isna().sum()

,0
SO2,0
CO,0
O3,0
NO2,0
PM10,0
PM25,0
기온(°C),0
강수량(mm),0
풍속(m/s),0
풍향(16방위),0


In [132]:
df_21.to_csv('df_21.csv', encoding='utf-8-sig')
df_22.to_csv('df_22.csv', encoding='utf-8-sig')